In [47]:
from ..memory import *
#访问长期记忆
load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    model='deepseek-v4-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={'thinking':{"type":'disabled'}},
)

In [48]:
#自定义state，在原来state只有messages的基础上增加额外的字段
class AccountState(AgentState):
    account_id:NotRequired[str]

class Account(BaseModel):
    account_id:str=Field(None,description='银行账户id')
    account_money:float=Field(0.0,description='账户余额',ge=0.0,le=9999.0)
    usr_name:str=Field('unknown',description="用户姓名")
@tool
#通过在创建agent时，使用state_schema参数传入自定义state，并在调用agent时传入state中的参数作为key,方法中使用runtime访问state中的作为key的参数
def save_money(runtime:ToolRuntime,money:float):
     '''存入银行账户金额
     Args:
         money:存入金额'''
     runtime.store.put(('account',),runtime.state['account_id'],{'money':money})
@tool
def query_money(runtime:ToolRuntime) -> dict:
    '''查询用户账户余额
        '''
    item=runtime.store.get(('account',),runtime.state['account_id'])
    return {
        'account_id':runtime.state['account_id'],
        'account_money':item.value['money'] if item else 0.0,
    }
store=InMemoryStore()
myagent=create_agent(
    model=model,
    store=store,
    state_schema=AccountState,
    response_format=ToolStrategy(Account),
    tools=[save_money,query_money],
    system_prompt='提取用户要求'
                  '如果用户要存钱，则调用保存方法'
                  '如果用户需要查询账户余额，则调用查询方法'
                  '返回结构化信息'
)

In [1]:

class InvalidActionError(Exception):
    def __init__(self, action):
        self.action = action
        message = f"{action}为无效操作"
        super().__init__(message)

def final_msg()->list[list[BaseMessage],str]:
    save_template=HumanMessagePromptTemplate.from_template('我是{name}，要存{money}元钱')
    query_template=HumanMessagePromptTemplate.from_template('我是{name},查询账户余额')
    action=input('请输入操作类型')
    if(action=='查询'):
        name=input('请输入姓名')
        query_msg=query_template.format_messages(name=name)
        msg=query_msg
        return [msg,name]
    elif(action=='存款'):
        name=input('请输入姓名')
        money=input('请输入存入金额')
        save_msg=save_template.format_messages(name=name,money=money)
        msg=save_msg
        return [msg,name]
    else:
        raise InvalidActionError(action)
def name2id(name:str)->str:
    if(name=='小明'):
        account_id='1001'
    elif(name=='小红'):
        account_id='1002'
    else:
        account_id='unknown'
    return account_id
print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
[msg1,name]=final_msg()
account_id=name2id(name)
response1=myagent.invoke({
    'messages':msg1,
    'account_id': account_id,
})
rprint(response1['messages'])
print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
[msg2,name]=final_msg()
account_id=name2id(name)
response2=myagent.invoke({
    'messages':msg2,
    'account_id': account_id,
})
rprint(response2["messages"])
print("=" * 30, '-> 第三个会话（线程） <-', "=" * 30)
[msg3,name]=final_msg()
account_id=name2id(name)
response3=myagent.invoke({
    'messages':msg3,
    'account_id': account_id,
})
rprint(response3['messages'])

NameError: name 'BaseMessage' is not defined

In [46]:
for item in store.search(("account",)):
    print(item)

Item(namespace=['account'], key='1001', value={'money': 500.0}, created_at='2026-07-28T09:25:57.618548+00:00', updated_at='2026-07-28T09:25:57.618548+00:00', score=None)
